<a href="https://colab.research.google.com/github/dilipbts/DL_Model_Movie_Review_Prediction_RNN/blob/main/sentiment_analysis_using_rnn.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ============================================================================
# SENTIMENT ANALYSIS WITH Bi‑LSTM (IMDB Dataset.csv)
# Google Colab – GPU Accelerated
# ============================================================================

# ------------------- 1. INSTALL / VERIFY DEPENDENCIES -------------------
# Colab comes with TensorFlow preinstalled, but we check anyway.
import subprocess
import sys
import os

try:
    import tensorflow as tf
    import pandas as pd
    import numpy as np
    from sklearn.model_selection import train_test_split
    from tensorflow.keras.preprocessing.text import Tokenizer
    from tensorflow.keras.preprocessing.sequence import pad_sequences
    from tensorflow.keras.models import Sequential
    from tensorflow.keras.layers import Embedding, Bidirectional, LSTM, Dense, Dropout
    from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint
except ImportError:
    print("Installing missing packages...")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "tensorflow", "pandas", "scikit-learn"])
    print("Restart runtime after installation.")
    # After installation, you may need to restart runtime; but we'll attempt reload.
    import tensorflow as tf
    import pandas as pd
    import numpy as np
    from sklearn.model_selection import train_test_split
    from tensorflow.keras.preprocessing.text import Tokenizer
    from tensorflow.keras.preprocessing.sequence import pad_sequences
    from tensorflow.keras.models import Sequential
    from tensorflow.keras.layers import Embedding, Bidirectional, LSTM, Dense, Dropout
    from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint

print("✅ All dependencies loaded.")
print("TensorFlow version:", tf.__version__)
print("GPU available:", tf.config.list_physical_devices('GPU'))

# ------------------- 2. UPLOAD YOUR DATASET -------------------
from google.colab import files
print("\n📤 Please upload your 'IMDB Dataset.csv' file.")
uploaded = files.upload()

# Get the uploaded filename (should be "IMDB Dataset.csv")
csv_file = list(uploaded.keys())[0]
print(f"✅ Uploaded: {csv_file}")

# ------------------- 3. CONFIGURATION -------------------
# All hyperparameters in one place
MAX_FEATURES = 20000      # Top 20k words
MAX_LEN = 200             # Pad/truncate reviews to 200 words
EMBEDDING_DIM = 128
LSTM_UNITS = 64
DROPOUT = 0.5
BATCH_SIZE = 64
EPOCHS = 10               # Early stopping will likely stop earlier
TRAIN_SPLIT = 0.8

# ------------------- 4. LOAD AND PREPROCESS DATA -------------------
print("\n[1/6] Loading CSV...")
df = pd.read_csv(csv_file)
print(f"   Loaded {len(df)} reviews.")

# Map sentiment to binary labels
df['sentiment'] = df['sentiment'].map({'positive': 1, 'negative': 0})

# Split into train/test
print("[2/6] Splitting data...")
X = df['review'].values
y = df['sentiment'].values
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=1 - TRAIN_SPLIT, random_state=42
)
print(f"   Train: {len(X_train)}, Test: {len(X_test)}")

# Tokenizer
print("[3/6] Fitting tokenizer...")
tokenizer = Tokenizer(num_words=MAX_FEATURES, oov_token="<OOV>")
tokenizer.fit_on_texts(X_train)
total_words = min(len(tokenizer.word_index) + 1, MAX_FEATURES + 1)

# Convert to sequences and pad
print("[4/6] Converting to sequences and padding...")
X_train_seq = tokenizer.texts_to_sequences(X_train)
X_test_seq = tokenizer.texts_to_sequences(X_test)

X_train_pad = pad_sequences(X_train_seq, maxlen=MAX_LEN, padding='post')
X_test_pad = pad_sequences(X_test_seq, maxlen=MAX_LEN, padding='post')

print(f"   Shapes: Train {X_train_pad.shape}, Test {X_test_pad.shape}")

# ------------------- 5. BUILD THE MODEL -------------------
print("[5/6] Building Bi‑LSTM model...")
model = Sequential([
    Embedding(total_words, EMBEDDING_DIM, input_length=MAX_LEN),
    Bidirectional(LSTM(LSTM_UNITS, dropout=DROPOUT, return_sequences=False)),
    Dense(64, activation='relu'),
    Dropout(0.5),
    Dense(1, activation='sigmoid')
])

model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
model.summary()

# ------------------- 6. TRAIN THE MODEL -------------------
print("[6/6] Training... (GPU will speed this up)")
callbacks = [
    EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True, verbose=1),
    ModelCheckpoint('sentiment_model.h5', save_best_only=True, monitor='val_loss', verbose=1)
]

history = model.fit(
    X_train_pad, y_train,
    batch_size=BATCH_SIZE,
    epochs=EPOCHS,
    validation_data=(X_test_pad, y_test),
    callbacks=callbacks,
    verbose=1
)

# Save final model
model.save('sentiment_model.h5')

# ------------------- 7. EVALUATE -------------------
print("\n" + "="*50)
print("EVALUATION ON TEST SET")
loss, acc = model.evaluate(X_test_pad, y_test, verbose=0)
print(f"Test Accuracy: {acc*100:.2f}%")
print(f"Test Loss: {loss:.4f}")
print("="*50)

# ------------------- 8. SHOW SAMPLE PREDICTIONS -------------------
print("\nSAMPLE PREDICTIONS (first 5 test reviews):")
print("-"*50)

# Decode function to show text snippets
reverse_word_index = {v: k for k, v in tokenizer.word_index.items()}
def decode_review(sequence):
    return ' '.join([reverse_word_index.get(i, '?') for i in sequence if i != 0])

for i in range(5):
    review_pad = X_test_pad[i].reshape(1, -1)
    prob = model.predict(review_pad, verbose=0)[0][0]
    pred = "Positive" if prob >= 0.5 else "Negative"
    true = "Positive" if y_test[i] == 1 else "Negative"
    snippet = decode_review(X_test_pad[i])[:150] + "..."
    print(f"\nExample {i+1}:")
    print(f"  True: {true}  |  Pred: {pred}  (confidence: {prob:.3f})")
    print(f"  Review snippet: {snippet}")

# ------------------- 9. DOWNLOAD MODEL (optional) -------------------
from google.colab import files
print("\n📥 Download the trained model? (run the next cell if you want to download)")
# To download, uncomment the lines below:
# files.download('sentiment_model.h5')
# files.download('tokenizer.pickle')   # If you want to save the tokenizer, uncomment and add code to pickle it.

# Save tokenizer for later use
import pickle
with open('tokenizer.pickle', 'wb') as f:
    pickle.dump(tokenizer, f)
files.download('tokenizer.pickle')

print("\n✅ All done! You can now use the model for inference.")

✅ All dependencies loaded.
TensorFlow version: 2.20.0
GPU available: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]

📤 Please upload your 'IMDB Dataset.csv' file.


Saving 1661-0_cleaned.txt to 1661-0_cleaned.txt
✅ Uploaded: 1661-0_cleaned.txt

[1/6] Loading CSV...
   Loaded 0 reviews.


KeyError: 'sentiment'

In [ ]:
# ==========================================
# INTERACTIVE SENTIMENT PREDICTOR (Manually Type Reviews)
# ==========================================
import tensorflow as tf
from tensorflow.keras.preprocessing.sequence import pad_sequences
import pickle

# Load the saved model and tokenizer
model = tf.keras.models.load_model('sentiment_model.h5')
with open('tokenizer.pickle', 'rb') as f:
    tokenizer = pickle.load(f)

MAX_LEN = 200

def predict_sentiment(text):
    seq = tokenizer.texts_to_sequences([text])
    padded = pad_sequences(seq, maxlen=MAX_LEN, padding='post')
    prob = model.predict(padded, verbose=0)[0][0]
    label = "Positive" if prob >= 0.5 else "Negative"
    return label, prob

# Interactive loop
print("\n" + "="*50)
print("🎬 MOVIE REVIEW SENTIMENT ANALYZER")
print("Type a movie review and I'll tell you if it's Positive or Negative.")
print("Type 'exit' or 'quit' to stop.")
print("="*50)

while True:
    user_review = input("\n📝 Enter your review: ").strip()

    if user_review.lower() in ['exit', 'quit', '']:
        print("Goodbye! 👋")
        break

    if not user_review:
        continue

    label, prob = predict_sentiment(user_review)
    print(f"🔮 Sentiment: {label} (Confidence: {prob:.3f})")

FileNotFoundError: [Errno 2] Unable to synchronously open file (unable to open file: name = 'sentiment_model.h5', errno = 2, error message = 'No such file or directory', flags = 0, o_flags = 0)

In [ ]:
from google.colab import files
files.download('sentiment_model.h5')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
import tensorflow as tf
from google.colab import files

# Load your existing model
model = tf.keras.models.load_model('sentiment_model.h5')

# Save as legacy H5 format – just use .h5 extension
model.save('sentiment_model_compat.h5')   # <-- No save_format argument!

# Download
files.download('sentiment_model_compat.h5')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>